# Netflix Prize — Extended EDA

Standalone analyses (no model needed) that go beyond the basic EDA in the MF notebook:
1. Average **title length** of the top-5 movies per release year
2. **Comeback** movies — old titles whose ratings surge late in the window
3. Tastes of **heavy raters (>100)** vs **newbies (<10)**
4. CSV export: **top-25 highest-rated movies per month** (for an IMDB join)

Note: rating timestamps span ~2000-2005; movie *release years* go back decades — a
separate field. All analyses are vectorized with `np.bincount` so they run on the full
100M ratings. Reuses the same `ratings_cache.npz` as the model notebooks.

## 0. Setup + load

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR  = '/content/netflix'      # <-- EDIT ME
CACHE_NPZ = os.path.join(DATA_DIR, 'ratings_cache.npz')
CHUNK_ROWS = 5_000_000
SUBSAMPLE_FILES = [1, 2, 3, 4]

def parse_combined(path, chunksize=CHUNK_ROWS):
    us, ms, rs, ds = [], [], [], []; cur = 0
    for chunk in pd.read_csv(path, header=None, names=['user','rating','date'],
                             dtype={'user': str}, chunksize=chunksize):
        is_hdr = chunk['rating'].isna()
        mv = pd.to_numeric(chunk['user'].where(is_hdr).str.rstrip(':'),
                           errors='coerce').ffill().fillna(cur)
        r = ~is_hdr
        us.append(pd.to_numeric(chunk.loc[r,'user']).to_numpy(np.int32))
        ms.append(mv[r].to_numpy(np.int16)); rs.append(chunk.loc[r,'rating'].to_numpy(np.int8))
        ds.append(chunk.loc[r,'date'].to_numpy('datetime64[D]')); cur = int(mv.iloc[-1])
    return (np.concatenate(us), np.concatenate(ms), np.concatenate(rs), np.concatenate(ds))

if os.path.exists(CACHE_NPZ):
    z = np.load(CACHE_NPZ)
    raw_user, movie1, rating, date = z['user'], z['movie'], z['rating'], z['date']
else:
    pu,pm,pr,pd_ = [],[],[],[]
    for n in SUBSAMPLE_FILES:
        u,m,r,d = parse_combined(os.path.join(DATA_DIR, f'combined_data_{n}.txt'))
        pu.append(u); pm.append(m); pr.append(r); pd_.append(d)
    raw_user=np.concatenate(pu); movie1=np.concatenate(pm)
    rating=np.concatenate(pr); date=np.concatenate(pd_)
    np.savez(CACHE_NPZ, user=raw_user, movie=movie1, rating=rating, date=date)

movie = (movie1.astype(np.int32) - 1)            # 0-based
n_movies = int(movie.max()) + 1
_, user = np.unique(raw_user, return_inverse=True); user = user.astype(np.int32)
n_users = int(user.max()) + 1
rating_f = rating.astype(np.float32)
print(f'{len(rating):,} ratings | {n_users:,} users | {n_movies:,} movies')
print('rating dates:', date.min(), '->', date.max())

titles = pd.read_csv(os.path.join(DATA_DIR,'movie_titles.csv'), header=None,
                     encoding='latin-1', on_bad_lines='skip',
                     names=['movie_id','year','title'], usecols=[0,1,2])
titles['year'] = pd.to_numeric(titles['year'], errors='coerce')
# per-movie tables aligned to 0-based index
mcount = np.bincount(movie, minlength=n_movies)
msum   = np.bincount(movie, weights=rating_f, minlength=n_movies)
mmean  = np.divide(msum, mcount, out=np.zeros_like(msum), where=mcount>0)
info = titles.set_index('movie_id')
year_of   = info['year'].reindex(np.arange(1, n_movies+1)).to_numpy()
title_arr = info['title'].reindex(np.arange(1, n_movies+1)).to_numpy()

## 1. Average title length of the top-5 movies per release year

For each release year, take the 5 most-rated movies and average their title length.

In [ ]:
tl = pd.DataFrame({'year': year_of, 'count': mcount, 'title': title_arr})
tl['title_len'] = tl['title'].str.len()
tl['n_words']   = tl['title'].str.split().str.len()
tl = tl[tl['year'].notna() & tl['title'].notna()]
top5 = tl.sort_values('count', ascending=False).groupby('year').head(5)
byyear = top5.groupby('year').agg(title_len=('title_len','mean'),
                                  words=('n_words','mean'), n=('count','size'))
byyear = byyear[byyear['n'] >= 5]                 # years with a full top-5

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
ax[0].plot(byyear.index, byyear['title_len'], marker='.')
ax[0].set_title('Avg title length (chars) of top-5 movies / release year'); ax[0].set_xlabel('year')
ax[1].plot(byyear.index, byyear['words'], marker='.', color='C1')
ax[1].set_title('Avg title length (words)'); ax[1].set_xlabel('year')
plt.tight_layout(); plt.show()
print(byyear.round(2).tail(12))

## 2. Comeback movies — old titles that surged late in the window

Netflix ratings span ~2000-2005, so an *old* film rated heavily here is being watched
20+ years after release. A **comeback** = old release year + a real rating base early in
the window + a strong surge in the second half + high average rating. `surge = late/early`.

> We can't see original box-office here (no external data), so 'failure -> comeback' is a
> proxy. Joining IMDB/box-office data (see Section 4) would let you confirm the flop part.

In [ ]:
mo = date.astype('datetime64[M]')
half = mo.min() + (mo.max() - mo.min()) // 2
early = mo < half; late = ~early
ec = np.bincount(movie[early], minlength=n_movies)
lc = np.bincount(movie[late],  minlength=n_movies)
surge = lc / np.maximum(ec, 1)
cb = pd.DataFrame({'movie_id': np.arange(1, n_movies+1), 'year': year_of,
                   'title': title_arr, 'n': mcount, 'avg': mmean.round(3),
                   'early': ec, 'late': lc, 'surge': surge.round(2)})
cb = cb[(cb['year'] < 1985) & (cb['early'] >= 20) & (cb['surge'] >= 3) & (cb['avg'] >= 3.7)]
print('Top comeback candidates (old, well-rated, accelerating):')
print(cb.sort_values('surge', ascending=False)
        [['year','title','n','avg','early','late','surge']].head(20).to_string(index=False))

## 3. Tastes of heavy raters (>100 ratings) vs newbies (<10)

Popularity is normalized **within** each group (share of the group's users who rated the
movie), so the comparison isn't dominated by group size. Surfaces what cinephiles vs
casual users actually watch.

In [ ]:
ucounts = np.bincount(user, minlength=n_users)
row_heavy = ucounts[user] > 100
row_newb  = ucounts[user] < 10
n_heavy = int((ucounts > 100).sum()); n_newb = int((ucounts < 10).sum())

def group_top(mask, n_group, k=12, min_ratings=50):
    c = np.bincount(movie[mask], minlength=n_movies)
    s = np.bincount(movie[mask], weights=rating_f, minlength=n_movies)
    m = np.divide(s, c, out=np.zeros_like(s), where=c>0)
    keep = np.where(c >= min_ratings)[0]
    top = keep[np.argsort(-(c[keep] / n_group))][:k]
    return pd.DataFrame({'title': title_arr[top], 'year': year_of[top],
                         'raters': c[top], 'reach_%': (100*c[top]/n_group).round(2),
                         'avg': m[top].round(2)})
print(f'HEAVY raters ({n_heavy:,}) — favorites:')
print(group_top(row_heavy, n_heavy).to_string(index=False))
print(f'\nNEWBIES ({n_newb:,}) — favorites:')
print(group_top(row_newb, n_newb).to_string(index=False))

## 4. Export: top-25 highest-rated movies per month -> CSV

For each month, the 25 movies with the highest average rating among those with at least
`MIN_RATINGS` ratings that month. Saved to `DATA_DIR/top25_per_month.csv` — join it to
IMDB on `(title, year)` for language / genre / theme analysis.

On Colab, download with: `from google.colab import files; files.download(out_path)`.

In [ ]:
MIN_RATINGS = 50
month_codes, month_idx = np.unique(mo, return_inverse=True)
M = len(month_codes)
key  = month_idx.astype(np.int64) * n_movies + movie
cnt  = np.bincount(key, minlength=M*n_movies).reshape(M, n_movies)
ssum = np.bincount(key, weights=rating_f, minlength=M*n_movies).reshape(M, n_movies)
mean = np.divide(ssum, cnt, out=np.zeros_like(ssum), where=cnt>0)

rows = []
for mi in range(M):
    valid = np.where(cnt[mi] >= MIN_RATINGS)[0]
    if len(valid) == 0:
        continue
    order = valid[np.argsort(-mean[mi, valid])][:25]   # ties broken by index
    for rank, mid in enumerate(order, 1):
        rows.append((str(month_codes[mi]), rank, mid + 1,
                     round(float(mean[mi, mid]), 4), int(cnt[mi, mid])))
out = pd.DataFrame(rows, columns=['month','rank','movie_id','avg_rating','n_ratings'])
out = out.merge(titles[['movie_id','year','title']], on='movie_id', how='left')
out = out[['month','rank','movie_id','title','year','n_ratings','avg_rating']]
out_path = os.path.join(DATA_DIR, 'top25_per_month.csv')
out.to_csv(out_path, index=False)
print(f'wrote {len(out):,} rows ({M} months) -> {out_path}')
out.head(10)

## Notes for the report

- **Title length**: feeds EDA (15%) and 'interesting observations'. Pair with a
  popularity trend to argue whether snappy titles correlate with reach.
- **Comebacks**: the surge metric + the IMDB join (box office vs. Netflix-era ratings)
  is a strong 'innovation' angle.
- **Heavy vs newbie**: motivates cold-start (newbies cluster on a few recent hits) and
  explains why personalization helps power users more.
- **top25_per_month.csv**: the IMDB join unlocks language / genre / theme breakdowns the
  raw dataset can't give.